# EEG-Based Alzheimer's Disease Detection
## Notebook 01 — Dataset Exploration

**Dataset:** OpenNeuro ds004504 (AHEPA cohort)  
**Researcher:** Hardik Thapar, Chitkara University, India  
**Supervisor:** Prof. Chang Kang-Ming, NKUST, Taiwan  
**Date:** March 2026

---

### What we're doing in this notebook

This is the first hands-on step of the research. Before building any model, we need to 
understand exactly what the data looks like — how many subjects, what the EEG signals 
look like, and whether the key biomarker we identified in the literature (DTABR) actually 
shows a difference between Alzheimer's patients and healthy controls.

The dataset has 88 subjects split into three groups:
- **AD** (Alzheimer's Disease) — 36 subjects, labelled 'A' in the data
- **FTD** (Frontotemporal Dementia) — 23 subjects, labelled 'F'  
- **HC** (Healthy Controls) — 29 subjects, labelled 'C'

We are using the **preprocessed files** from the `derivatives` folder — these have already 
been bandpass filtered (0.5–45 Hz), artifact-corrected (ASR), and had eye/jaw artifacts 
removed (ICA). This is the standard starting point used by all major papers on this dataset.

In [7]:
# Install MNE-Python — the standard library for EEG analysis.
# Kaggle doesn't have it pre-installed so we install it at session start.
!pip install mne -q

import mne
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Suppress MNE's verbose output — we only want to see warnings and errors
mne.set_log_level('WARNING')

print(f"MNE version    : {mne.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"Pandas version : {pd.__version__}")
print("All libraries ready.")

MNE version    : 1.11.0
NumPy version  : 2.0.2
Pandas version : 2.3.3
All libraries ready.


## Step 1 — Load the Participant Labels

The file `participants.tsv` tells us which subject belongs to which group.
- Group **A** = Alzheimer's Disease
- Group **C** = Healthy Control  
- Group **F** = Frontotemporal Dementia

It also contains Age, Gender, and MMSE score (cognitive test — lower = worse).
We load this first so we can attach labels to every subject's EEG data.

In [8]:
# Download ds004504 directly from OpenNeuro S3 using the AWS CLI.
# This bypasses the Git Annex requirement and works directly in Kaggle.
# Only downloads the derivatives folder (preprocessed files) — about 1.2GB.

import subprocess
import os

# Install AWS CLI if not available
subprocess.run(['pip', 'install', 'awscli', '-q'], check=True)

# Create the target directory
os.makedirs('/kaggle/working/ds004504/derivatives', exist_ok=True)

# Download just the derivatives folder from OpenNeuro's public S3 bucket
# No AWS account needed — this bucket is publicly accessible
result = subprocess.run([
    'aws', 's3', 'sync',
    's3://openneuro.org/ds004504/derivatives/',
    '/kaggle/working/ds004504/derivatives/',
    '--no-sign-request',
    '--region', 'us-east-1'
], capture_output=True, text=True)

print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)
if result.returncode == 0:
    print("\nDownload complete.")
else:
    print("Error:", result.stderr[-1000:])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 20.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
aiobotocore 3.3.0 requires botocore<1.42.71,>=1.42.62, but you have botocore 1.42.73 which is incompatible.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.4 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.5 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.5 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.5 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.5 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.5 MiB/s) with 1 file(s) remaining
Completed 2.7 GiB/2.7 GiB (110.5 MiB/s) with 1 

In [9]:
# Download the participants.tsv file which has the diagnostic labels.
result2 = subprocess.run([
    'aws', 's3', 'cp',
    's3://openneuro.org/ds004504/participants.tsv',
    '/kaggle/working/ds004504/participants.tsv',
    '--no-sign-request',
    '--region', 'us-east-1'
], capture_output=True, text=True)

print(result2.stdout)
if result2.returncode == 0:
    print("participants.tsv downloaded.")
else:
    print("Error:", result2.stderr)

Completed 1.7 KiB/1.7 KiB (5.6 KiB/s) with 1 file(s) remaining
download: s3://openneuro.org/ds004504/participants.tsv to ds004504/participants.tsv

participants.tsv downloaded.


In [10]:
# The participants.tsv file is the key metadata file.
# It maps each subject ID to their diagnosis group, age, gender, and MMSE score.
# Without this file we have EEG signals but no labels — useless for classification.

data_path = Path("/kaggle/working/ds004504/")

# Load participants info
participants_file = data_path / "participants.tsv"
participants = pd.read_csv(participants_file, sep='\t')

# Clean up column names just in case
participants.columns = participants.columns.str.strip()

# Map short codes to full group names for readability
group_map = {'A': 'Alzheimer', 'F': 'FTD', 'C': 'Healthy Control'}
participants['group_name'] = participants['Group'].map(group_map)

print("=" * 55)
print("PARTICIPANT OVERVIEW")
print("=" * 55)
print(participants[['participant_id', 'Group', 'group_name', 'Age', 'Gender', 'MMSE']].to_string(index=False))
print("\n" + "=" * 55)
print("GROUP COUNTS")
print("=" * 55)
print(participants['group_name'].value_counts().to_string())
print(f"\nTotal subjects: {len(participants)}")

PARTICIPANT OVERVIEW
participant_id Group      group_name  Age Gender  MMSE
       sub-001     A       Alzheimer   57      F    16
       sub-002     A       Alzheimer   78      F    22
       sub-003     A       Alzheimer   70      M    14
       sub-004     A       Alzheimer   67      F    20
       sub-005     A       Alzheimer   70      M    22
       sub-006     A       Alzheimer   61      F    14
       sub-007     A       Alzheimer   79      F    20
       sub-008     A       Alzheimer   62      M    16
       sub-009     A       Alzheimer   77      F    23
       sub-010     A       Alzheimer   69      M    20
       sub-011     A       Alzheimer   71      M    22
       sub-012     A       Alzheimer   63      M    18
       sub-013     A       Alzheimer   64      F    20
       sub-014     A       Alzheimer   77      M    14
       sub-015     A       Alzheimer   61      M    18
       sub-016     A       Alzheimer   68      F    14
       sub-017     A       Alzheimer   61   

## Step 2 — Locate the Preprocessed EEG Files

The dataset has two versions of each subject's EEG:
- **Raw files** in `/sub-XXX/eeg/` — original unprocessed recordings
- **Preprocessed files** in `/derivatives/sub-XXX/eeg/` — filtered and artifact-cleaned

We use the **preprocessed (derivatives)** files. These have already been:
1. Bandpass filtered at 0.5–45 Hz
2. Artifact Subspace Reconstruction (ASR) applied
3. ICA run to remove eye and jaw movement artifacts

This is the same preprocessed version used by Vo et al. (2025) and Khan et al. (2025).

In [11]:
# We look inside the derivatives folder for all .set files.
# Each subject has one .set file containing their complete preprocessed EEG recording.
# The filename format is: sub-XXX_task-eyesclosed_eeg.set

derivatives_path = data_path / "derivatives"

eeg_files = []
for sub_folder in sorted(derivatives_path.iterdir()):
    if sub_folder.is_dir() and sub_folder.name.startswith('sub-'):
        eeg_folder = sub_folder / 'eeg'
        if eeg_folder.exists():
            for f in eeg_folder.glob('*.set'):
                eeg_files.append({
                    'subject_id': sub_folder.name,
                    'filepath': str(f)
                })

eeg_df = pd.DataFrame(eeg_files)
print(f"Total EEG files found: {len(eeg_df)}")
print(f"\nFirst 5 files:")
print(eeg_df.head().to_string(index=False))

# Merge with participant labels
eeg_df = eeg_df.merge(
    participants[['participant_id', 'Group', 'group_name', 'Age', 'Gender', 'MMSE']],
    left_on='subject_id',
    right_on='participant_id',
    how='left'
)

print(f"\nFiles matched with labels: {eeg_df['Group'].notna().sum()} / {len(eeg_df)}")

Total EEG files found: 88

First 5 files:
subject_id                                                                         filepath
   sub-001 /kaggle/working/ds004504/derivatives/sub-001/eeg/sub-001_task-eyesclosed_eeg.set
   sub-002 /kaggle/working/ds004504/derivatives/sub-002/eeg/sub-002_task-eyesclosed_eeg.set
   sub-003 /kaggle/working/ds004504/derivatives/sub-003/eeg/sub-003_task-eyesclosed_eeg.set
   sub-004 /kaggle/working/ds004504/derivatives/sub-004/eeg/sub-004_task-eyesclosed_eeg.set
   sub-005 /kaggle/working/ds004504/derivatives/sub-005/eeg/sub-005_task-eyesclosed_eeg.set

Files matched with labels: 88 / 88


## Step 3 — Load One Subject and Inspect the Raw EEG Signal

Before processing all 88 subjects, we load one subject and understand what 
the data actually looks like. This tells us:
- How many channels are there
- What the sampling rate is  
- How long the recording is
- What the raw brain signal looks like visually

We pick one AD subject, one FTD subject, and one healthy control 
so we can visually compare their signals side by side.

In [12]:
# Load the first AD subject to inspect the EEG structure.
# raw.info contains all the metadata we need to understand the recording.

# Pick one subject from each group for comparison
ad_subject  = eeg_df[eeg_df['Group'] == 'A'].iloc[0]
ftd_subject = eeg_df[eeg_df['Group'] == 'F'].iloc[0]
hc_subject  = eeg_df[eeg_df['Group'] == 'C'].iloc[0]

# Load the AD subject first
raw = mne.io.read_raw_eeglab(ad_subject['filepath'], preload=True, verbose=False)

print("=" * 55)
print(f"SUBJECT: {ad_subject['subject_id']}  |  Group: Alzheimer's Disease")
print("=" * 55)
print(f"Sampling rate     : {raw.info['sfreq']} Hz")
print(f"Number of channels: {len(raw.ch_names)}")
print(f"Channel names     : {raw.ch_names}")
print(f"Recording duration: {raw.times[-1]:.1f} seconds  ({raw.times[-1]/60:.1f} minutes)")
print(f"Total data points : {len(raw.times):,} samples")
print(f"Data shape        : {raw.get_data().shape}  (channels × timepoints)")
print(f"MMSE score        : {ad_subject['MMSE']}  (out of 30 — lower = more severe)")

SUBJECT: sub-001  |  Group: Alzheimer's Disease
Sampling rate     : 500.0 Hz
Number of channels: 19
Channel names     : ['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']
Recording duration: 599.8 seconds  (10.0 minutes)
Total data points : 299,900 samples
Data shape        : (19, 299900)  (channels × timepoints)
MMSE score        : 16  (out of 30 — lower = more severe)


## Step 4 — Visualise the Raw EEG Signal

We plot the first 10 seconds of EEG for all 19 channels.
Each line represents one electrode on the scalp.
The waves you see are the brain's electrical activity in microvolts.

In Alzheimer's patients, these waves tend to be dominated by slower frequencies 
(delta and theta bands) compared to healthy controls who show more alpha activity.

In [13]:
# Plot the raw EEG signal for the first 10 seconds.
# We offset each channel vertically so they don't overlap — this is standard in EEG visualisation.

fig, axes = plt.subplots(3, 1, figsize=(16, 14))
subjects_to_plot = [
    (ad_subject,  'Alzheimer\'s Disease (AD)', '#D32F2F'),
    (ftd_subject, 'Frontotemporal Dementia (FTD)', '#F57C00'),
    (hc_subject,  'Healthy Control (HC)', '#388E3C')
]

for ax, (subject, label, color) in zip(axes, subjects_to_plot):
    raw_sub = mne.io.read_raw_eeglab(subject['filepath'], preload=True, verbose=False)
    data, times = raw_sub.get_data(return_times=True)
    time_mask = times <= 10  # first 10 seconds

    # Normalise each channel by its std dev so amplitudes are comparable
    for i in range(len(raw_sub.ch_names)):
        std = data[i].std()
        if std > 0:
            normalised = data[i, time_mask] / std
        else:
            normalised = data[i, time_mask]
        offset = i * 3  # vertical spacing between channels
        ax.plot(times[time_mask], normalised + offset,
                linewidth=0.5, alpha=0.8, color=color)

    ax.set_yticks([i * 3 for i in range(len(raw_sub.ch_names))])
    ax.set_yticklabels(raw_sub.ch_names, fontsize=8)
    ax.set_title(f"{label}  —  Subject: {subject['subject_id']}  |  MMSE: {subject['MMSE']}",
                 fontsize=12, fontweight='bold', color=color)
    ax.set_xlabel("Time (seconds)", fontsize=10)
    ax.grid(True, alpha=0.2)

plt.suptitle("Raw EEG Signal — First 10 Seconds (All 19 Channels)",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("raw_eeg_comparison.png", dpi=120, bbox_inches='tight')
plt.show()
print("Plot saved: raw_eeg_comparison.png")

/tmp/ipykernel_55/2950774948.py:12: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw_sub = mne.io.read_raw_eeglab(subject['filepath'], preload=True, verbose=False)


Plot saved: raw_eeg_comparison.png


## Step 5 — Power Spectral Density (PSD)

Power Spectral Density shows how much power (energy) exists at each frequency.
This is the most important plot for Alzheimer's EEG research.

The five frequency bands we care about:
| Band  | Range      | Expected change in AD |
|-------|------------|----------------------|
| Delta | 0.5 – 4 Hz | **Increases** in AD  |
| Theta | 4 – 8 Hz   | **Increases** in AD  |
| Alpha | 8 – 13 Hz  | **Decreases** in AD  |
| Beta  | 13 – 30 Hz | **Decreases** in AD  |
| Gamma | 30 – 45 Hz | Disrupted in AD      |

The DTABR ratio — **(Delta + Theta) / (Alpha + Beta)** — goes UP as Alzheimer's gets worse.

In [14]:
# Compare Power Spectral Density across all three groups.
# We average PSD across all channels and all subjects in each group.
# This shows the group-level difference in brain activity — the EEG signature of AD.

def compute_mean_psd(filepath, fmin=0.5, fmax=45):
    """Load an EEG file and compute mean PSD across all channels."""
    raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
    psd = raw.compute_psd(fmin=fmin, fmax=fmax, method='welch', verbose=False)
    return psd.freqs, psd.get_data().mean(axis=0)

print("Computing PSD for all 88 subjects — this takes about 2-3 minutes...")

group_psds = {'A': [], 'F': [], 'C': []}

for _, row in eeg_df.iterrows():
    try:
        freqs, mean_psd = compute_mean_psd(row['filepath'])
        group_psds[row['Group']].append(mean_psd)
    except Exception as e:
        print(f"  Skipped {row['subject_id']}: {e}")

# Average across subjects within each group
avg_psd = {
    'Alzheimer (AD)':       np.mean(group_psds['A'], axis=0),
    'FTD':                  np.mean(group_psds['F'], axis=0),
    'Healthy Control (HC)': np.mean(group_psds['C'], axis=0),
}

# Plot
fig, ax = plt.subplots(figsize=(14, 7))
colors = {'Alzheimer (AD)': '#D32F2F', 'FTD': '#F57C00', 'Healthy Control (HC)': '#388E3C'}

for group_label, psd_vals in avg_psd.items():
    ax.semilogy(freqs, psd_vals, linewidth=2, label=group_label,
                color=colors[group_label])

# Shade frequency bands
band_colors = {'Delta\n0.5–4Hz': (0.5, 4, '#FF6B6B'), 'Theta\n4–8Hz': (4, 8, '#FFA500'),
               'Alpha\n8–13Hz': (8, 13, '#4CAF50'), 'Beta\n13–30Hz': (13, 30, '#2196F3'),
               'Gamma\n30–45Hz': (30, 45, '#9C27B0')}

for label, (fmin, fmax, c) in band_colors.items():
    ax.axvspan(fmin, fmax, alpha=0.12, color=c, label=label)

ax.set_xlabel("Frequency (Hz)", fontsize=13)
ax.set_ylabel("Power Spectral Density (µV²/Hz)", fontsize=13)
ax.set_title("Average EEG Power Spectrum — AD vs FTD vs Healthy Control",
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("psd_group_comparison.png", dpi=120, bbox_inches='tight')
plt.show()
print("PSD comparison plot saved.")

Computing PSD for all 88 subjects — this takes about 2-3 minutes...


/tmp/ipykernel_55/2152700633.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/tmp/ipykernel_55/2152700633.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/tmp/ipykernel_55/2152700633.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
/tmp/ipykernel_55/2152700633.py:7: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)


PSD comparison plot saved.


## Step 6 — Compute DTABR for Every Subject

DTABR = **(Delta + Theta power) / (Alpha + Beta power)**

This is the single most important EEG biomarker for Alzheimer's.
- Higher DTABR = more slow-wave activity = more AD-like brain
- We expect: **AD > FTD > Healthy Control**

We compute this for all 88 subjects and build a summary table.
This table will be the foundation for everything that follows.

In [15]:
# Compute DTABR for every subject and build a clean summary table.
# This is the key analytical step — we're generating our first real research result.

def get_band_power(psd_data, freqs, fmin, fmax):
    """
    Extract power in a specific frequency band using the trapezoidal integration rule.
    Returns average across all channels.
    """
    band_mask = (freqs >= fmin) & (freqs <= fmax)
    if band_mask.sum() == 0:
        return 0.0
    return np.trapz(psd_data[:, band_mask], freqs[band_mask], axis=1).mean()

print("Computing DTABR for all subjects...")
print("-" * 65)

records = []

for _, row in eeg_df.iterrows():
    try:
        raw = mne.io.read_raw_eeglab(row['filepath'], preload=True, verbose=False)
        psd = raw.compute_psd(fmin=0.5, fmax=45, method='welch', verbose=False)
        f   = psd.freqs
        p   = psd.get_data()  # shape: (channels, frequencies)

        delta = get_band_power(p, f, 0.5, 4)
        theta = get_band_power(p, f, 4,   8)
        alpha = get_band_power(p, f, 8,   13)
        beta  = get_band_power(p, f, 13,  30)
        gamma = get_band_power(p, f, 30,  45)

        dtabr = (delta + theta) / (alpha + beta) if (alpha + beta) > 0 else np.nan

        records.append({
            'subject_id'  : row['subject_id'],
            'group'       : row['Group'],
            'group_name'  : row['group_name'],
            'age'         : row['Age'],
            'mmse'        : row['MMSE'],
            'delta'       : round(delta, 8),
            'theta'       : round(theta, 8),
            'alpha'       : round(alpha, 8),
            'beta'        : round(beta,  8),
            'gamma'       : round(gamma, 8),
            'DTABR'       : round(dtabr, 4),
            'duration_min': round(raw.times[-1] / 60, 1),
        })

        print(f"  {row['subject_id']}  |  {row['group_name']:<20}  |  MMSE: {row['MMSE']:>4}  |  DTABR: {dtabr:.4f}")

    except Exception as e:
        print(f"  Skipped {row['subject_id']}: {e}")

results_df = pd.DataFrame(records)
results_df.to_csv("subject_dtabr_summary.csv", index=False)

print("\n" + "=" * 65)
print(f"Processed: {len(results_df)} subjects  |  Saved to subject_dtabr_summary.csv")

Computing DTABR for all subjects...
-----------------------------------------------------------------


/tmp/ipykernel_55/4240836743.py:12: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return np.trapz(psd_data[:, band_mask], freqs[band_mask], axis=1).mean()


  sub-001  |  Alzheimer             |  MMSE:   16  |  DTABR: 21.7202
  sub-002  |  Alzheimer             |  MMSE:   22  |  DTABR: 10.0601
  sub-003  |  Alzheimer             |  MMSE:   14  |  DTABR: 7.7618


/tmp/ipykernel_55/4240836743.py:21: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(row['filepath'], preload=True, verbose=False)
/tmp/ipykernel_55/4240836743.py:21: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(row['filepath'], preload=True, verbose=False)


  sub-004  |  Alzheimer             |  MMSE:   20  |  DTABR: 19.5622
  sub-005  |  Alzheimer             |  MMSE:   22  |  DTABR: 18.4871


...

/tmp/ipykernel_55/4240836743.py:21: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(row['filepath'], preload=True, verbose=False)


  sub-081  |  FTD                   |  MMSE:   18  |  DTABR: 22.2576
  sub-082  |  FTD                   |  MMSE:   27  |  DTABR: 14.9033
  sub-083  |  FTD                   |  MMSE:   20  |  DTABR: 23.3188
  sub-084  |  FTD                   |  MMSE:   24  |  DTABR: 15.2750
  sub-085  |  FTD                   |  MMSE:   26  |  DTABR: 13.2240


/tmp/ipykernel_55/4240836743.py:21: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(row['filepath'], preload=True, verbose=False)
/tmp/ipykernel_55/4240836743.py:21: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(row['filepath'], preload=True, verbose=False)
/tmp/ipykernel_55/4240836743.py:21: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(row['filepath'], preload=True, verbose=False)


  sub-086  |  FTD                   |  MMSE:   26  |  DTABR: 7.9490
  sub-087  |  FTD                   |  MMSE:   24  |  DTABR: 18.3970
  sub-088  |  FTD                   |  MMSE:   24  |  DTABR: 9.9481

Processed: 88 subjects  |  Saved to subject_dtabr_summary.csv


/tmp/ipykernel_55/4240836743.py:21: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(row['filepath'], preload=True, verbose=False)


## Step 7 — DTABR Comparison Across Groups

Now we compare the DTABR values across the three groups.
If the biomarker works as the literature says, we should see:
**AD has the highest DTABR → FTD in the middle → Healthy Controls the lowest**

This would validate our dataset before we invest weeks building a model on it.

In [16]:
# Compare DTABR across groups with a boxplot.
# A boxplot shows the distribution — median, spread, and outliers.
# We expect the AD box to sit clearly higher than the HC box.

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

group_order  = ['Alzheimer', 'FTD', 'Healthy Control']
group_colors = ['#D32F2F', '#F57C00', '#388E3C']

# Left plot — DTABR boxplot
ax = axes[0]
data_by_group = [results_df[results_df['group_name'] == g]['DTABR'].dropna().values
                 for g in group_order]

bp = ax.boxplot(data_by_group, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))

for patch, color in zip(bp['boxes'], group_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

# Add individual data points on top
for i, (data, color) in enumerate(zip(data_by_group, group_colors)):
    x = np.random.normal(i + 1, 0.06, size=len(data))
    ax.scatter(x, data, alpha=0.6, color=color, s=30, zorder=3)

ax.set_xticklabels(group_order, fontsize=11)
ax.set_ylabel("DTABR  (Delta+Theta) / (Alpha+Beta)", fontsize=11)
ax.set_title("DTABR by Diagnostic Group", fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Right plot — DTABR vs MMSE scatter
ax = axes[1]
for group, color, name in zip(['A', 'F', 'C'], group_colors, group_order):
    subset = results_df[results_df['group'] == group]
    ax.scatter(subset['mmse'], subset['DTABR'],
               color=color, label=name, s=60, alpha=0.8)

ax.set_xlabel("MMSE Score (cognitive test — lower = more severe)", fontsize=11)
ax.set_ylabel("DTABR", fontsize=11)
ax.set_title("DTABR vs Cognitive Score (MMSE)", fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle("Key EEG Biomarker Analysis — ds004504", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("dtabr_analysis.png", dpi=120, bbox_inches='tight')
plt.show()
print("DTABR analysis plot saved.")

DTABR analysis plot saved.


## Step 8 — Summary Statistics

Final numbers from the exploration.
This is what we'll report in the Google Doc and to Prof. Chang.

In [18]:
# Print clean summary statistics grouped by diagnosis.
# These numbers go directly into the research log.

print("=" * 65)
print("DATASET EXPLORATION — FINAL SUMMARY")
print("=" * 65)

summary_stats = results_df.groupby('group_name').agg(
    n_subjects    = ('subject_id', 'count'),
    mean_age      = ('age',   lambda x: f"{x.mean():.1f} ± {x.std():.1f}"),
    mean_mmse     = ('mmse',  lambda x: f"{x.mean():.1f} ± {x.std():.1f}"),
    mean_duration = ('duration_min', lambda x: f"{x.mean():.1f} min"),
    mean_dtabr    = ('DTABR', lambda x: f"{x.mean():.4f} ± {x.std():.4f}"),
    min_dtabr     = ('DTABR', 'min'),
    max_dtabr     = ('DTABR', 'max'),
).reset_index()

print(summary_stats.to_string(index=False))

print("\n" + "=" * 65)
print("KEY FINDING")
print("=" * 65)
ad_mean  = results_df[results_df['group'] == 'A']['DTABR'].mean()
ftd_mean = results_df[results_df['group'] == 'F']['DTABR'].mean()
hc_mean  = results_df[results_df['group'] == 'C']['DTABR'].mean()

print(f"Average DTABR — Alzheimer's : {ad_mean:.4f}")
print(f"Average DTABR — FTD         : {ftd_mean:.4f}")
print(f"Average DTABR — Healthy     : {hc_mean:.4f}")

if ad_mean > hc_mean:
    print("\n Confirmed: AD shows higher DTABR than Healthy Controls.")
    print("  This validates the dataset and our biomarker choice.")
    print("  We are ready to move to the preprocessing pipeline.")
else:
    print("\n Unexpected result — check if derivatives folder was used correctly.")

print("\nNext step: Preprocessing pipeline — bandpass filter, epoching, feature extraction.")

DATASET EXPLORATION — FINAL SUMMARY
     group_name  n_subjects   mean_age  mean_mmse mean_duration       mean_dtabr  min_dtabr  max_dtabr
      Alzheimer          36 66.4 ± 7.9 17.8 ± 4.5      13.5 min 15.6449 ± 5.1318     5.8362    26.2987
            FTD          23 63.7 ± 8.2 22.2 ± 2.6      12.0 min 14.2077 ± 5.3689     4.6372    23.3188
Healthy Control          29 67.9 ± 5.4 30.0 ± 0.0      13.9 min 10.9807 ± 4.5865     3.3848    20.2991

KEY FINDING
Average DTABR — Alzheimer's : 15.6449
Average DTABR — FTD         : 14.2077
Average DTABR — Healthy     : 10.9807

 Confirmed: AD shows higher DTABR than Healthy Controls.
  This validates the dataset and our biomarker choice.
  We are ready to move to the preprocessing pipeline.

Next step: Preprocessing pipeline — bandpass filter, epoching, feature extraction.
